# NetraEdge — Complete Model Training
## Face Recognition + Liveness Detection

**Steps:**
1. Runtime > Change runtime type > **GPU (T4)**
2. Runtime > **Run all**
3. Download the two .onnx files from the output

In [ ]:
#@title 1. Setup — Install deps + Check GPU
!pip install -q onnx onnxscript tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import os, time, random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name())
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > GPU')

In [ ]:
#@title 2. Define MobileFaceNet
class DWSep(nn.Module):
    def __init__(self, ic, oc, st=1):
        super().__init__()
        self.dw = nn.Conv2d(ic, ic, 3, st, 1, groups=ic, bias=False)
        self.b1 = nn.BatchNorm2d(ic)
        self.pw = nn.Conv2d(ic, oc, 1, bias=False)
        self.b2 = nn.BatchNorm2d(oc)
    def forward(self, x):
        return F.relu(self.b2(self.pw(F.relu(self.b1(self.dw(x))))))

class SE(nn.Module):
    def __init__(self, ch, r=4):
        super().__init__()
        m = max(ch // r, 8)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, m), nn.ReLU(True),
            nn.Linear(m, ch), nn.Sigmoid())
    def forward(self, x):
        return x * self.se(x).unsqueeze(-1).unsqueeze(-1)

class MB(nn.Module):
    def __init__(self, ic, oc, st=1):
        super().__init__()
        m = ic * 2
        self.ex = nn.Sequential(
            nn.Conv2d(ic, m, 1, bias=False), nn.BatchNorm2d(m), nn.ReLU(True))
        self.dw = DWSep(m, oc, st)
        self.se = SE(oc)
        self.res = (st == 1 and ic == oc)
    def forward(self, x):
        o = self.se(self.dw(self.ex(x)))
        return o + x if self.res else o

class MobileFaceNet(nn.Module):
    def __init__(self, ed=128):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 2, 1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(True))
        self.blk = nn.Sequential(
            MB(64, 64), MB(64, 128, 2), MB(128, 128),
            MB(128, 256, 2), MB(256, 256), MB(256, 256),
            MB(256, 512, 2), MB(512, 512), MB(512, 512))
        self.fin = nn.Sequential(
            nn.Conv2d(512, 512, 3, groups=512, bias=False),
            nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, ed, 1, bias=False), nn.BatchNorm2d(ed))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    def forward(self, x):
        x = self.fin(self.blk(self.stem(x)))
        return F.normalize(x.view(x.size(0), -1), p=2, dim=1)

m = MobileFaceNet(128)
nparams = sum(p.numel() for p in m.parameters())
print('MobileFaceNet:', nparams, 'params')

In [ ]:
#@title 3. Dataset — Synthetic faces (50 identities x 20 images)
class FaceDataset(Dataset):
    def __init__(self, n_id=50, n_img=20, augment=True):
        self.samples = []
        self.augment = augment
        for i in range(n_id):
            rng = np.random.RandomState(i * 1000)
            base = rng.rand(112, 112, 3).astype(np.float32) * 0.3 + 0.35
            base[35:50, 35:55, :] *= 0.7
            base[35:50, 60:80, :] *= 0.7
            base[45:65, 50:65, :] += 0.05
            base[70:85, 40:75, 0] += 0.1
            for j in range(n_img):
                face = base + rng.randn(112, 112, 3).astype(np.float32) * 0.05
                self.samples.append((np.clip(face, 0, 1), i))
        print(len(self.samples), 'images,', n_id, 'identities')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        face, label = self.samples[idx]
        arr = face.copy()
        if self.augment:
            if random.random() > 0.5:
                arr = np.clip(arr + np.random.uniform(-0.08, 0.08), 0, 1)
            if random.random() > 0.5:
                arr = np.flip(arr, axis=1).copy()
        t = torch.from_numpy(arr).permute(2, 0, 1).float()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return (t - mean) / std, label

tr = FaceDataset(50, 20, True)
va = FaceDataset(50, 20, False)
tr_dl = DataLoader(tr, 32, shuffle=True, num_workers=2, pin_memory=True)
va_dl = DataLoader(va, 32, shuffle=False, num_workers=2)
print('Train:', len(tr), 'Val:', len(va))

In [ ]:
#@title 4. Train Face Recognition (10 epochs, ~2 min on T4)
from tqdm import tqdm

model = MobileFaceNet(128).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
best = 0

for ep in range(10):
    t0 = time.time()
    model.train()
    tc = tt = 0
    for x, y in tqdm(tr_dl, desc='Ep ' + str(ep+1) + '/10'):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        tc += (out.argmax(1) == y).sum().item()
        tt += y.size(0)
    model.eval()
    vc = vt = 0
    with torch.no_grad():
        for x, y in va_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            vc += (model(x).argmax(1) == y).sum().item()
            vt += y.size(0)
    ta = 100*tc/tt
    va_acc = 100*vc/vt
    elapsed = int(time.time()-t0)
    scheduler.step()
    print('Ep ' + str(ep+1) + ': Train ' + str(round(ta,1)) + '% Val ' + str(round(va_acc,1)) + '% ' + str(elapsed) + 's')
    if va_acc > best:
        best = va_acc
        torch.save(model.state_dict(), 'rec_best.pt')
print('Best:', round(best,1), '%')

In [ ]:
#@title 5. Export Face Recognition to ONNX
model.cpu().eval()
torch.onnx.export(model, torch.randn(1, 3, 112, 112),
    'face_recognition.onnx', input_names=['input'],
    output_names=['output'], opset_version=13, dynamo=False)
sz = os.path.getsize('face_recognition.onnx') / 1e6
print('face_recognition.onnx:', round(sz, 1), 'MB')
from google.colab import files
files.download('face_recognition.onnx')

---
## Liveness Detection Training

In [ ]:
#@title 6. LivenessCNN + Dataset + Train (10 epochs, ~1 min on T4)
class LivBlock(nn.Module):
    def __init__(self, ic, oc, st=1):
        super().__init__()
        self.dw = nn.Conv2d(ic, ic, 3, st, 1, groups=ic, bias=False)
        self.b1 = nn.BatchNorm2d(ic)
        self.pw = nn.Conv2d(ic, oc, 1, bias=False)
        self.b2 = nn.BatchNorm2d(oc)
    def forward(self, x):
        return F.relu(self.b2(self.pw(F.relu(self.b1(self.dw(x))))))

class LivenessCNN(nn.Module):
    def __init__(self, nc=3, dp=0.3):
        super().__init__()
        self.feat = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(True),
            LivBlock(32, 64), LivBlock(64, 128, 2),
            LivBlock(128, 256, 2), LivBlock(256, 256, 2))
        self.cls = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(dp), nn.Linear(256, 64), nn.ReLU(True),
            nn.Dropout(dp * 0.5), nn.Linear(64, nc))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    def forward(self, x):
        return self.cls(self.feat(x))

class LivDataset(Dataset):
    def __init__(self, n=3000):
        self.n = n
        self.bases = []
        for i in range(100):
            rng = np.random.RandomState(i * 777)
            face = rng.rand(112, 112, 3).astype(np.float32) * 0.4 + 0.3
            face[35:50, 35:55, :] *= 0.7
            face[35:50, 60:80, :] *= 0.7
            face[70:85, 40:75, 0] += 0.1
            self.bases.append(face)
    def __len__(self): return self.n
    def __getitem__(self, idx):
        label = idx % 3
        base = self.bases[idx % len(self.bases)]
        arr = base.copy()
        rng = np.random.RandomState(idx)
        if label == 1:
            arr = np.clip(arr * 1.05 + 0.02, 0, 1)
            k = np.ones(5) / 5
            arr = np.stack([np.convolve(arr[:,:,c].flatten(), k, mode='same').reshape(112,112) for c in range(3)], -1)
        elif label == 2:
            for y in range(0, 112, 2):
                arr[y,:,:] *= 0.88
            arr[:,:,2] *= 1.05
            arr = arr * 0.85 + 0.08
        arr = np.clip(arr + rng.randn(112, 112, 3).astype(np.float32) * 0.02, 0, 1)
        if random.random() > 0.5:
            arr = np.flip(arr, axis=1).copy()
        t = torch.from_numpy(arr).permute(2, 0, 1).float()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
        return (t - mean) / std, label

tr2 = LivDataset(3000)
va2 = LivDataset(600)
tr2_dl = DataLoader(tr2, 64, shuffle=True, num_workers=2, pin_memory=True)
va2_dl = DataLoader(va2, 64, shuffle=False, num_workers=2)

livmodel = LivenessCNN(3).to(DEVICE)
w = torch.tensor([1.0, 1.5, 1.5]).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
optimizer = torch.optim.AdamW(livmodel.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
best = 0

for ep in range(10):
    t0 = time.time()
    livmodel.train()
    tc = tt = 0
    for x, y in tqdm(tr2_dl, desc='Ep ' + str(ep+1) + '/10'):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(livmodel(x), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(livmodel.parameters(), 5.0)
        optimizer.step()
        tc += (livmodel(x).argmax(1) == y).sum().item()
        tt += y.size(0)
    livmodel.eval()
    vc = vt = 0
    with torch.no_grad():
        for x, y in va2_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            vc += (livmodel(x).argmax(1) == y).sum().item()
            vt += y.size(0)
    ta = 100*tc/tt
    va_acc = 100*vc/vt
    elapsed = int(time.time()-t0)
    scheduler.step()
    print('Ep ' + str(ep+1) + ': Train ' + str(round(ta,1)) + '% Val ' + str(round(va_acc,1)) + '% ' + str(elapsed) + 's')
    if va_acc > best:
        best = va_acc
        torch.save(livmodel.state_dict(), 'liv_best.pt')
print('Best:', round(best,1), '%')

In [ ]:
#@title 7. Export Liveness to ONNX
livmodel.cpu().eval()
torch.onnx.export(livmodel, torch.randn(1, 3, 112, 112),
    'liveness_detector.onnx', input_names=['input'],
    output_names=['output'], opset_version=13, dynamo=False)
sz = os.path.getsize('liveness_detector.onnx') / 1e6
print('liveness_detector.onnx:', round(sz, 1), 'MB')
from google.colab import files
files.download('liveness_detector.onnx')

In [ ]:
#@title 8. Summary
rec_sz = os.path.getsize('face_recognition.onnx') / 1e6
liv_sz = os.path.getsize('liveness_detector.onnx') / 1e6
total = rec_sz + liv_sz
print('=== NetraEdge Models ===')
print('face_recognition.onnx:', round(rec_sz, 1), 'MB')
print('liveness_detector.onnx:', round(liv_sz, 1), 'MB')
print('Total:', round(total, 1), 'MB (target: <20MB)')
print('')
print('Place both files in: F:/PROJECTS/NetraEdge/models/')